# Your first isolated coding-agent environment

This notebook walks from a fresh machine to a reusable Docker Sandbox template with a small network allowlist. You do not need prior Docker knowledge.

Read each section and copy its Bash commands into the **same host terminal**, in order. These are terminal examples, not executable Python cells. No notebook kernel or Python packages are needed.

The tutorial prepares the environment only. The MCP server separately handles task assignment, independent checkouts, autonomous execution, and patch retrieval.

## 1. Understand what you are creating

A coding agent can edit files and execute programs. A **sandbox** gives it a separate environment so those actions are contained. Docker Sandboxes uses a microVM: a lightweight virtual machine with its own filesystem and network boundary.

A **template** is a saved filesystem image used to create more sandboxes. A **network policy** determines which external hosts a sandbox can contact. Network rules live outside the image and must be configured for every new sandbox.

For now we share no host workspace and start no agent. We create an empty Codex environment, allow selected documentation sites and OpenAI endpoints, then save it.

Sources: [Docker isolation](https://docs.docker.com/ai/sandboxes/security/isolation/) and [templates](https://docs.docker.com/ai/sandboxes/usage/).

## 2. Install Docker Sandboxes on the host

For this macOS walkthrough, use Apple silicon with macOS 14 or later and Homebrew. Docker Desktop is not required for the standalone `sbx` CLI. For Windows or Ubuntu, follow the platform-specific [installation guide](https://docs.docker.com/ai/sandboxes/install/), then return here and use a Bash terminal.

Run these installation commands once:

```bash
brew trust docker/tap
brew install docker/tap/sbx
```

### Which account do you need?

A free Docker account is enough for this local walkthrough, including commercial or professional use. You can use a Docker account registered with your work email; a paid enterprise account is not required just to sign in or run local sandboxes. “Work account” here means a Docker account, rather than an account with your AI provider or an employer login that has not been registered with Docker.

Docker requires sign-in to associate sandboxes with a verified identity. The account requirements depend on what you use:

| Use | Account or payment requirement |
| --- | --- |
| This tutorial: standalone CLI and local sandbox compute | Free Docker account; commercial and professional use included |
| Local allowlist rules configured in this tutorial | No paid organization governance subscription required |
| Centrally managed organization policies, enforced sign-in, and audit logs | Separate paid Docker governance subscription |
| Docker-managed cloud sandbox compute | Active Docker Agentic Platform pay-as-you-go subscription |
| AI model usage inside the sandbox | Separate model-provider authentication and any applicable charges |

The commands in this notebook use local sandboxes, without `--cloud`. Docker sign-in does not provide AI model access or model credits. If your employer manages Docker access, its sign-in and organization policies still apply.

These requirements were checked on 6 November 2026 against Docker's [FAQ](https://docs.docker.com/ai/sandboxes/faq/) and [cloud prerequisites](https://docs.docker.com/ai/sandboxes/cloud/).

Check that installation worked, then sign in to Docker in the browser opened by the second command:

```bash
sbx --version
sbx login
```

Use your Docker account for this browser sign-in. You should see a CLI version and complete Docker sign-in. These host installation and login steps are separate from the restricted network available inside a sandbox.

## 3. Start with locked-down networking

On a fresh installation, initialize **Locked Down**, whose CLI name is `deny-all`. This makes destinations unavailable unless a policy explicitly permits them.

```bash
sbx policy init deny-all
sbx policy ls
```

This is a global setting for your local Docker Sandboxes installation. Before adding any sandbox-specific network rules, a typical listing looks like this:

```text
POLICY             SOURCE   APPLIES TO   SUMMARY
default-deny-all   local    all          network: deny when no rule decides
local-policy       local    all          filesystem read: 1 allow; filesystem write: 1 allow
```

Read the columns as follows:

- **POLICY** names the policy.
- **SOURCE** shows where the policy comes from; `local` means this machine's local configuration.
- **APPLIES TO** identifies its scope; `all` means all local sandboxes.
- **SUMMARY** describes the kinds of access controlled by its rules.

The two rows above mean:

- **default-deny-all** denies outbound network access whenever no explicit rule decides the request. This is the expected locked-down network default. It still allows us to add specific domain exceptions later.
- **local-policy** contains one filesystem read allow rule and one filesystem write allow rule. These counts describe filesystem permissions, not network exceptions. Their presence does not mean internet access is enabled. This summary does not show the paths covered by those rules.

Exact formatting and filesystem rule counts may vary with CLI version and configuration. The checkpoint is a deny-by-default network policy without broader network allow rules. With the example listing above, you can continue to step 4; no policy reset is needed.

If initialization says a policy already exists, inspect `sbx policy ls` instead of initializing again. If it matches the locked-down setup above, continue. If it shows a different default or additional network policies, inspect their rules before proceeding; do not assume they restrict access to this tutorial's allowlist. Organization policy may also affect access.

Changing an existing setup requires deliberate policy management. `sbx policy reset` removes local rules and stops running sandboxes, so it is not part of this fresh-install tutorial.

Reference: [network policies](https://docs.docker.com/ai/governance/local/network-rules/).

## 4. Choose names and create the source sandbox

Keep this terminal open: later commands reuse these variables. Change the values if you want different names. Use a new sandbox name for a new run.

```bash
sandbox_name=agent-template-source
template_name=autonomous-agent:v1

sbx create --name "$sandbox_name" --skills off --static-mcp none codex
```

This creates a Codex environment without starting an agent session. Omitting a workspace path keeps host project files out of it; shared skills and static MCP connections are disabled.

Creation may download the base image through the host runtime. The agent's network allowlist is a separate policy. If creation fails, resolve the reported error before continuing; rerunning with the same name may report that it already exists.

## 5. Add the approved domains

Enable the destinations in [WHITELISTED_DOMAINS.md](WHITELISTED_DOMAINS.md). Each hostname is explicit; subdomains are not included implicitly.

Python packages use `pypi.org` and `files.pythonhosted.org`; Node.js packages use `registry.npmjs.org`. Mozilla provides web documentation. The ChatGPT domains support Codex subscription-based model access; OAuth browser sign-in itself happens on the host. Package registries and Docker Hub use GET/HEAD HTTP rules, with one POST exception for npm vulnerability audits. PyPI permits its simple index, package file paths, and version-specific vulnerability lookups for pip-audit; npm permits metadata and tarball downloads.

### Inspect the permissions inherited from Codex

The locked-down preset denies requests only when no rule allows them. Creating a Codex sandbox also installs kit rules that can allow GitHub, npm, and operating-system package hosts. Our allowlist does not replace those rules.

Inspect the actual installed kit before configuring access:

```bash
sbx policy ls "$sandbox_name" --source kit --decision allow --wide
```

The kit's host list can change between releases. Compare its resources with the approved destinations below. Add every unwanted inherited domain or pattern to `denied_domains` before running the function. The list below covers unapproved GitHub hosting domains and common system package mirrors; it is a starting list, not a complete inventory of every kit version. Preserve any port suffix from a resource you copy, or omit the port to deny that host on all ports.

A local deny overrides kit allows. Do not deny all hosts with `**`: subsequent allow rules cannot reopen a denied destination. Likewise, do not deny `*.docker.com` if you intend to retain `docs.docker.com`.

```bash
allowed_domains=(
  openai.com
  platform.openai.com
  api.openai.com
  chatgpt.com
  www.chatgpt.com
  docs.python.org
  nodejs.org
  dev.mysql.com
  mongodb.com
  www.mongodb.com
  postgresql.org
  www.postgresql.org
  docs.docker.com
  developer.mozilla.org
)

# Extend this list using the installed kit's resource listing above.
denied_domains=(
  githubassets.com
  '*.githubassets.com'
  github.io
  '*.github.io'
  archive.ubuntu.com
  '*.archive.ubuntu.com'
  security.ubuntu.com
  ports.ubuntu.com
  esm.ubuntu.com
  changelogs.ubuntu.com
  keyserver.ubuntu.com
  deb.debian.org
  security.debian.org
  download.docker.com
)

apply_network_rules() {
  local target_sandbox="$1"
  for domain in "${denied_domains[@]}"; do
    sbx policy deny network --sandbox "$target_sandbox" "$domain"
  done
  # Keep GitHub SSH blocked while allowing inspected HTTPS reads.
  sbx policy deny network --sandbox "$target_sandbox" github.com:22
  sbx policy deny network --sandbox "$target_sandbox" ssh.github.com
  for domain in "${allowed_domains[@]}"; do
    sbx policy allow network --sandbox "$target_sandbox" "$domain:443"
  done

  for domain in github.com raw.githubusercontent.com codeload.github.com api.github.com; do
    sbx policy allow network --sandbox "$target_sandbox" "$domain:443" --method GET,HEAD --path '/**'
  done

  sbx policy allow network --sandbox "$target_sandbox" pypi.org:443 --method GET,HEAD --path '/simple/**'
  sbx policy allow network --sandbox "$target_sandbox" pypi.org:443 --method GET --path '/pypi/*/*/json'
  sbx policy allow network --sandbox "$target_sandbox" files.pythonhosted.org:443 --method GET,HEAD --path '/packages/**'
  sbx policy allow network --sandbox "$target_sandbox" registry.npmjs.org:443 --method GET,HEAD --path '/**'
  sbx policy allow network --sandbox "$target_sandbox" registry.npmjs.org:443 --method POST --path '/-/npm/v1/security/advisories/bulk'

  sbx policy allow network --sandbox "$target_sandbox" registry-1.docker.io:443 --method GET,HEAD --path '/v2/**'
  sbx policy allow network --sandbox "$target_sandbox" auth.docker.io:443 --method GET,HEAD --path '/token' --query 'service=registry.docker.io'
}
```

### Review inherited permissions on approved hosts

An inherited whole-host allow on GitHub or an approved registry can also bypass our method/path allow rules. For example, a kit allow for `registry.npmjs.org` permits more than our audit exception.

Host denies are suitable for unwanted destinations, but cannot be used to remove a broad GitHub or registry allow while keeping download exceptions. If the kit grants broad access to GitHub or a registry, use a custom kit with that broad permission removed, or organization policy that enforces the narrower rules. Kit rules are read-only in the local policy CLI.

Do not treat the GitHub and registry restrictions below as enforced while conflicting broad allows remain. HTTP deny rules can block selected operations, but adding narrow HTTP allows alone cannot constrain an inherited whole-host allow.

### Apply and spot-check the policy

```bash
apply_network_rules "$sandbox_name"
sbx policy ls "$sandbox_name" --wide

# GitHub reads should be allowed; write methods should be denied.
sbx policy check http --sandbox "$sandbox_name" --method GET https://github.com/
sbx policy check http --sandbox "$sandbox_name" --method HEAD https://raw.githubusercontent.com/python/cpython/main/README.rst
sbx policy check http --sandbox "$sandbox_name" --method POST https://github.com/
sbx policy check http --sandbox "$sandbox_name" --method PUT https://api.github.com/repos/example/example/contents/probe
sbx policy check http --sandbox "$sandbox_name" --method DELETE https://api.github.com/repos/example/example/contents/probe

# A package download and the audit exception should be allowed.
sbx policy check http --sandbox "$sandbox_name" --method GET https://pypi.org/simple/requests/
sbx policy check http --sandbox "$sandbox_name" --method POST https://registry.npmjs.org/-/npm/v1/security/advisories/bulk

# A Docker Hub upload should be denied.
sbx policy check http --sandbox "$sandbox_name" --method POST https://registry-1.docker.io/v2/library/ubuntu/blobs/uploads/
```

These are policy evaluations, not actual HTTP requests. Expect GitHub GET/HEAD, the package download, and the audit exception to be allowed; GitHub POST/PUT/DELETE and the Docker upload should be denied. If an expected denial is allowed, stop and resolve the matching inherited or global rule before running an agent.

### Try real requests from inside the sandbox

Run these commands from the host terminal. `sbx exec` runs curl inside the sandbox, where its outbound policy applies. Run each command separately so an expected failure does not prevent the next example.

First confirm an allowed download works:

```bash
sbx exec "$sandbox_name" curl --silent --show-error --fail --max-time 15 --output /dev/null --write-out 'HTTP %{http_code}' https://pypi.org/simple/requests/
```

Expect HTTP 200. Also verify GitHub GET and HEAD:

```bash
sbx exec "$sandbox_name" curl --silent --show-error --fail --max-time 15 --output /dev/null --write-out 'HTTP %{http_code}' https://github.com/python/cpython/blob/main/README.rst
sbx exec "$sandbox_name" curl --head --silent --show-error --fail --max-time 15 https://raw.githubusercontent.com/python/cpython/main/README.rst
```

Expect successful responses. Then try three forbidden requests:

```bash
# A forbidden GitHub method; no credentials or upload payload.
sbx exec "$sandbox_name" curl --include --silent --show-error --fail-with-body --max-time 15 --request POST https://github.com/

# An unapproved path on an allowed host: unversioned PyPI JSON.
sbx exec "$sandbox_name" curl --include --silent --show-error --fail-with-body --max-time 15 https://pypi.org/pypi/requests/json

# An unapproved method on an approved path: POST to the read-only index.
sbx exec "$sandbox_name" curl --include --silent --show-error --fail-with-body --max-time 15 --request POST https://pypi.org/simple/requests/
```

Expect these requests to be blocked and curl to return a nonzero status. The exact response depends on the proxy and client; it may be an HTTP denial or a connection/proxy error. These probes send no credentials or upload payloads.

Confirm the denials in the host-side logs:

```bash
sbx policy log "$sandbox_name" --limit 20
```

Look for blocked decisions for those destinations and requests. A curl error alone does not prove policy enforcement: DNS failures, timeouts, missing curl, and upstream HTTP errors can also fail. An upstream 404 or 405 is not evidence of a sandbox block. Investigate unexpected results before proceeding.

Docker pulls use GET to retrieve content and HEAD to check its existence. Normal pushes need write methods, which these rules omit. HEAD has no normal upload payload, but its URL and headers still transmit data; GET/HEAD restrictions are not a complete guarantee against data leakage. Documentation and OpenAI destinations in `allowed_domains` retain whole-host access.

The npm rules allow audit submissions while blocking normal package publishing. Once an npm project is available inside a worker, run `npm audit --registry=https://registry.npmjs.org` there. Audits normally submit package names and versions, including potentially private package names, to npm. The POST exception permits any request body to that endpoint; it does not validate audit payloads. Legacy audit fallback endpoints remain blocked. PyPI's unversioned JSON API and other unmatched paths remain excluded. These download rules do not limit which package versions can be fetched or make downloaded code trustworthy.

### Install and use pip-audit

Use pip-audit's PyPI service, which queries `GET /pypi/{package}/{version}/json`. This uses the existing PyPI domain without enabling OSV or GitHub access. Package names and versions are sent to PyPI in URL paths.

Install the tool inside the Linux source sandbox before saving the template in step 7 (requires Python 3.10 or newer):

```bash
sbx exec "$sandbox_name" python3 -m pip install pip-audit
```

After project dependencies have been installed in a worker, audit that environment from the host terminal:

```bash
sbx exec "$worker_name" python3 -m pip_audit --vulnerability-service pypi
```

Use the project's Python interpreter instead of `python3` if dependencies live in a virtual environment, and install pip-audit there too. Auditing the source sandbox now would check only its current environment, not future project dependencies.

For a fully resolved requirements file with every transitive dependency pinned, an alternative command inside the worker is:

```bash
python3 -m pip_audit --vulnerability-service pypi --no-deps --disable-pip -r requirements.txt
```

The last command avoids pip-based dependency resolution. Reports identify known vulnerabilities, not arbitrary malicious packages. Review skipped dependencies and errors; exit code 1 can indicate vulnerabilities were found, so read the report.

HTTP-restricted traffic must use the sandbox's HTTP-inspecting proxy path; a client using an unsupported transport may be blocked. Actual image downloads may redirect to additional content-delivery hosts, which remain blocked unless explicitly approved. GitHub permits GET/HEAD on all paths at `github.com`, `raw.githubusercontent.com`, `codeload.github.com`, and `api.github.com`. This includes source and archive downloads, not just documentation. GitHub Pages and asset hosts remain blocked. Downloaded code can access worker files and approved network destinations; URLs and headers can disclose data. Review worker patches before integration.

Step 8 reuses `apply_network_rules` so new workers receive the explicit denies as well as the allow rules. Inspect each new worker's effective kit permissions too; the template does not replace kit policy.

Sources: [policy precedence](https://docs.docker.com/ai/sandboxes/governance/concepts/), [Codex kit permissions](https://docs.docker.com/learn/content/run-an-agent-safely/), [Docker registry API](https://docs.docker.com/reference/api/registry/latest/), [registry authentication](https://docs.docker.com/reference/api/registry/auth/), [sandbox HTTP rules](https://docs.docker.com/ai/governance/local/network-rules/), [npm audit](https://docs.npmjs.com/cli/commands/npm-audit/), and [pip-audit](https://pypi.org/project/pip-audit/).

## 6. Verify access before running an agent

List the rules, then ask the policy engine to evaluate representative destinations:

```bash
sbx policy ls "$sandbox_name"
sbx policy check network --sandbox "$sandbox_name" api.openai.com:443
sbx policy check network --sandbox "$sandbox_name" docs.python.org:443
sbx policy check http --sandbox "$sandbox_name" --method GET https://github.com/
sbx policy check network --sandbox "$sandbox_name" github.com:22
sbx policy check network --sandbox "$sandbox_name" example.com:443
```

Expect OpenAI and Python documentation to be allowed; GitHub GET should be allowed; GitHub SSH and the unlisted example domain should be denied. These commands evaluate policy, not website availability or API authentication.

If an expected result differs, resolve it before continuing. Broader global or organization rules can affect effective access. The selected checks are useful spot checks; review the complete rule listing too.

### Configure Codex authentication before saving the template

`sbx login` authenticates with Docker. Codex needs a separate OpenAI credential.

Choose **one** method in the host terminal:

**ChatGPT sign-in (OAuth):**

```bash
sbx secret set openai --oauth
```

Complete the browser sign-in using your OpenAI account with Codex access. Docker stores the resulting tokens on the host. Local OAuth credentials are global, so workers on this host can reuse them.

**Alternatively, an OpenAI API key:**

```bash
sbx secret set openai
```

Paste the key into the prompt. API usage uses the associated API billing. This command also stores a global credential. For a key restricted to the source sandbox, use `sbx secret set openai --sandbox "$sandbox_name"` instead, then configure a separate scoped key for each worker after creating it.

Inspect stored secret entries without printing their values:

```bash
sbx secret ls
```

Expect an OpenAI credential entry. This confirms storage, not a successful model request. Do not configure both methods unless you intend to replace or change the selected credential.

Credentials stay in macOS Keychain or Windows Credential Manager. On Linux without a desktop keyring, Docker may use a permission-protected file. The host proxy supplies authentication at runtime; real credentials are not saved into the template. Do not copy `~/.codex/auth.json` or manually place keys in the sandbox.

A template moved to another host needs credentials configured on that host. A new worker on this host can reuse the global credential, subject to any credential-use approval shown by Docker.

Sources: [Docker Codex authentication](https://docs.docker.com/ai/sandboxes/agents/codex/), [credential storage](https://docs.docker.com/ai/sandboxes/configuration/credentials), and [official OpenAI authentication documentation](https://developers.openai.com/codex/auth/).


## 7. Save the reusable template

With the tools installed, network checks complete, and OpenAI authentication configured on the host in step 6, save the source sandbox's filesystem:

```bash
sbx template save "$sandbox_name" "$template_name"
sbx template ls
```

Expect `autonomous-agent:v1` (or your chosen name) in the template listing.

The image contains the Codex environment and tools installed in step 5. It does not contain network policy, mounted workspaces, or host-managed OpenAI credentials. Each worker needs its own network rules and access to a runtime credential. Step 8 applies the rules and explains how to test Codex.


## 8. Create another sandbox from the template

This demonstrates reuse without starting an agent or sharing host files. Run step 5 first in the same Bash terminal to define `apply_network_rules`.

```bash
worker_name=agent-worker-demo
sbx create --name "$worker_name" --pull never --template "$template_name" --skills off --static-mcp none codex

apply_network_rules "$worker_name"

sbx policy ls "$worker_name"
sbx policy check network --sandbox "$worker_name" api.openai.com:443
sbx policy check http --sandbox "$worker_name" --method GET https://github.com/
sbx policy check http --sandbox "$worker_name" --method POST https://github.com/
sbx policy check network --sandbox "$worker_name" github.com:22
sbx policy check http --sandbox "$worker_name" --method HEAD https://registry-1.docker.io/v2/library/ubuntu/manifests/latest
sbx policy check http --sandbox "$worker_name" --method POST https://registry-1.docker.io/v2/library/ubuntu/blobs/uploads/
```

Expect OpenAI, GitHub GET, and Docker Hub HEAD allowed; GitHub POST, GitHub SSH, and Docker Hub POST denied. Repeat the real-request probes and confirm denials in this worker's policy logs before running its agent. Reapplying the rules is necessary because the template stores filesystem content, while policy belongs to the sandbox.

The main-agent workflow prepares an independent checkout, provides task input to an isolated worker, waits for completion, and retrieves a readable Git patch. Those orchestration steps are implemented by the MCP server, not this setup notebook.

### Optional: verify Codex can make a model request

The global OpenAI credential configured in step 6 is reusable here. If you chose a sandbox-scoped API key, configure the worker's key first with `sbx secret set openai --sandbox "$worker_name"`.

From the host terminal:

```bash
sbx run --name "$worker_name"
```

Review any credential-use approval Docker displays. In Codex, ask: `Reply with "Authentication works". Do not execute commands or modify files.`

A successful reply verifies model access and may consume model usage. Exit Codex with `/exit`. If access fails, distinguish missing credentials, account permissions, billing, and blocked destinations using the error and policy logs; a stored secret alone does not verify connectivity.

The reusable template remains separate from this worker session.


## 9. Optional cleanup

When you no longer need these demonstration sandboxes:

```bash
sbx rm "$worker_name"
sbx rm "$sandbox_name"
```

The saved template remains available. Remove it too if you no longer need it:

```bash
sbx template rm "$template_name"
```

Removing sandboxes deletes their internal files. Retrieve any future task results before removing a worker.
